# MPP training on Kaggle

1. **Add Data**: добавь датасет с `data_with_dates.csv` (или загрузи CSV в Input).
2. **Репо**: укажи ниже URL своего репо (или залей проект как датасет и поменяй путь).
3. Включи **GPU** в Settings → Accelerator.
4. Запусти все ячейки.

In [1]:
# Путь к данным: если добавил датасет с CSV — укажи имя датасета или оставь авто-поиск
import os
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

# Найти data_with_dates.csv в добавленных датасетах
def find_data_csv():
    for f in KAGGLE_INPUT.rglob("data_with_dates.csv"):
        return str(f)
    return None

DATA_CSV = find_data_csv()
if DATA_CSV is None:
    # Или укажи вручную, например: DATA_CSV = "/kaggle/input/my-dataset/data_with_dates.csv"
    raise FileNotFoundError(
        "Не найден data_with_dates.csv в /kaggle/input. Добавь датасет с этим файлом или задай DATA_CSV вручную."
    )
print("Data CSV:", DATA_CSV)

FileNotFoundError: Не найден data_with_dates.csv в /kaggle/input. Добавь датасет с этим файлом или задай DATA_CSV вручную.

In [ ]:
# Клонируем репо с кодом (замени на свой URL и ветку)
REPO_URL = "https://github.com/YOUR_USER/ML_project-football-.git"  # ← заменить
BRANCH = "attention_test"  # ветка для клонирования
REPO_DIR = KAGGLE_WORKING / "ML_project-football-"

if not REPO_DIR.exists() or not (REPO_DIR / "models").exists():
    import subprocess
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Репо уже есть:", REPO_DIR)

# Ставим зависимости
REQ = REPO_DIR / "requirements.txt"
if REQ.exists():
    import subprocess
    subprocess.run(["pip", "install", "-q", "-r", str(REQ)], check=True)

import sys
sys.path.insert(0, str(REPO_DIR))
ROOT = REPO_DIR
print("ROOT:", ROOT)

In [ ]:
# Подготовка: копируем CSV в dataset/ репо и создаём рабочие папки
import shutil

dataset_dir = REPO_DIR / "dataset"
dataset_dir.mkdir(parents=True, exist_ok=True)
dest_csv = dataset_dir / "data_with_dates.csv"
shutil.copy(DATA_CSV, dest_csv)
print("Скопировано в:", dest_csv)

(REPO_DIR / "notebooks").mkdir(parents=True, exist_ok=True)
(REPO_DIR / "notebooks" / "mpp_mini_output").mkdir(parents=True, exist_ok=True)

In [ ]:
# Запуск обучения (логика из run.py)
import os
os.chdir(REPO_DIR)

import pandas as pd
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

from data.preprocessing import preprocess_raw_csv, build_vocab_mappings
from data.dataset import MatchDatasetMPP, PreCollatedDataset
from data.collator import DataCollatorMPP
try:
    from data.collator import DataCollatorPreCollated
except ImportError:
    from torch.utils.data.dataloader import default_collate
    class DataCollatorPreCollated:
        def __call__(self, batch):
            return default_collate(batch)
from models.pretrain import MaskedPlayerModel
from training.trainer import build_training_args, build_trainer
from training.metrics import compute_metrics_mpp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

sample_path = REPO_DIR / "notebooks" / "train_sample_raw.csv"
output_dir = str(REPO_DIR / "notebooks" / "train_sample_processed")
Path(output_dir).mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(dest_csv)
df_raw.to_csv(sample_path, index=False)
df = preprocess_raw_csv(str(sample_path), output_dir)
vocab = build_vocab_mappings(df, output_dir)

print("Матчей:", df["match_id"].nunique(), "players_vocab (pad+1):", vocab["player_pad_token_id"] + 1)

In [ ]:
max_seq_length = 36
sample_batch_size = 256
repeat = 20
dev_ratio = 0.05
seed = 42

ds_full = MatchDatasetMPP(
    df,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)

collator = DataCollatorMPP(
    player_mask_token_id=vocab["player_mask_token_id"],
    mask_percentage=0.25,
)

def _collate_filter_none(batch):
    batch = [b for b in batch if b is not None]
    return collator(batch) if batch else None

dataloader_build = DataLoader(
    ds_full, batch_size=sample_batch_size, shuffle=True,
    collate_fn=_collate_filter_none, drop_last=True,
)

all_batches = []
for _ in range(repeat):
    for batch in dataloader_build:
        if batch is not None:
            all_batches.append(batch)

np.random.seed(seed)
n_batches = len(all_batches)
dev_size = max(1, int(n_batches * dev_ratio))
dev_idx = np.random.choice(n_batches, size=dev_size, replace=False)
train_idx = [i for i in range(n_batches) if i not in dev_idx]

train_batches = [all_batches[i] for i in train_idx]
dev_batches = [all_batches[i] for i in dev_idx]

train_dataset = PreCollatedDataset(train_batches)
eval_dataset = PreCollatedDataset(dev_batches)
collator_for_trainer = DataCollatorPreCollated()

print("Батчей:", n_batches, "train:", len(train_batches), "eval:", len(dev_batches))

In [ ]:
model = MaskedPlayerModel(
    embed_size=128,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=39,
    players_vocab_size=vocab["player_pad_token_id"],
    teams_vocab_size=vocab["team_pad_token_id"],
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
model = model.to(device)

output_dir = str(REPO_DIR / "notebooks" / "mpp_mini_output")
training_config = {
    "output_dir": output_dir,
    "num_train_epochs": 2000,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "learning_rate": 1e-4,
    "weight_decay": 0.0,
    "warmup_ratio": 0.0,
    "lr_scheduler_type": "linear",
    "logging_steps": 1000,
    "eval_strategy": "steps",
    "eval_steps": 1000,
    "save_strategy": "steps",
    "save_steps": 10000,
    "save_total_limit": 3,
    "report_to": "none",
    "seed": seed,
}

train_args = build_training_args(training_config)
trainer = build_trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics_mpp,
    data_collator=collator_for_trainer,
)
print("Параметров:", sum(p.numel() for p in model.parameters()))

In [ ]:
final_dir = REPO_DIR / "notebooks" / "mpp_mini_output" / "final_model"
try:
    trainer.train()
finally:
    trainer.save_model(str(final_dir))
    print("Модель сохранена:", final_dir)

if trainer.state.log_history:
    pd.DataFrame(trainer.state.log_history).to_csv(
        REPO_DIR / "notebooks" / "mpp_mini_output" / "metrics.csv", index=False
    )
    print("Метрики сохранены в metrics.csv")